### Predicción real con clientes nuevos

In [1]:
import joblib
import json
import pandas as pd

# 1. Cargar el modelo guardado
modelo_cargado = joblib.load('../models/random_forest_cancelaciones.pkl')

# 2. Cargar los nuevos clientes desde JSON 
with open('../data/mongo_clients_new.json', 'r') as f:
    nuevos_datos = json.load(f)

# 3. Normalizar el JSON para convertirlo en un DataFrame
df_nuevos = pd.json_normalize(nuevos_datos)

df_nuevos = df_nuevos.rename(columns={
    'hotel.type':                         'hotel_type',
    'customer.type':                      'customer_type',
    'booking_info.adr':                   'adr',
    'booking_info.lead_time':             'lead_time',
    'booking_info.status':                'status',
    'booking_info.guests.adults':         'adults',
    'booking_info.guests.children':       'children',
    'booking_info.guests.babies':         'babies',
    'booking_info.deposit_type':          'deposit_type',
    'booking_info.previous_cancellations':'previous_cancellations',
    'booking_info.total_special_requests':'total_special_requests',
    'booking_info.stays_weekend':         'stays_weekend',
    'booking_info.stays_weekday':         'stays_weekday',
})


# 4. Seleccionar exactamente las mismas features que usó el modelo
features_num = ['lead_time', 'adr', 'adults', 'children', 'babies', 'previous_cancellations', 'total_special_requests', 'stays_weekend', 'stays_weekday']
features_cat = ['hotel_type', 'customer_type', 'deposit_type']
X_nuevos = df_nuevos[features_num + features_cat]

# 5. Predecir
predicciones  = modelo_cargado.predict(X_nuevos)
probabilidades = modelo_cargado.predict_proba(X_nuevos)

# 6. Mostrar resultados
print("=" * 60)
print("   PREDICCIONES PARA NUEVOS CLIENTES")
print("=" * 60)

for i, (pred, prob) in enumerate(zip(predicciones, probabilidades)):
    estado   = "CANCELA" if pred == 1 else "NO CANCELA"
    prob_pct = prob[pred] * 100
    cliente  = df_nuevos.iloc[i]
    print(f"\nCliente {i+1} (ID: {int(df_nuevos.iloc[i]['reservation_sql_id'])}): {estado} ({prob_pct:.1f}% confianza)")
    print(f"  → Hotel: {cliente['hotel_type']} | Lead time: {cliente['lead_time']} días")
    print(f"  → ADR: ${cliente['adr']} | Adultos: {int(cliente['adults'])}")
    print(f"  → Children: {int(cliente['children'])} | Babies: {int(cliente['babies'])}")
print("\n" + "=" * 60)

   PREDICCIONES PARA NUEVOS CLIENTES

Cliente 1 (ID: 9001): NO CANCELA (61.5% confianza)
  → Hotel: Resort Hotel | Lead time: 342 días
  → ADR: $0.0 | Adultos: 2
  → Children: 0 | Babies: 0

Cliente 2 (ID: 9002): NO CANCELA (73.7% confianza)
  → Hotel: City Hotel | Lead time: 10 días
  → ADR: $120.0 | Adultos: 1
  → Children: 0 | Babies: 0

Cliente 3 (ID: 9003): CANCELA (80.6% confianza)
  → Hotel: City Hotel | Lead time: 180 días
  → ADR: $85.0 | Adultos: 3
  → Children: 1 | Babies: 0

Cliente 4 (ID: 9004): CANCELA (75.2% confianza)
  → Hotel: Resort Hotel | Lead time: 5 días
  → ADR: $250.0 | Adultos: 2
  → Children: 0 | Babies: 0

